In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
admin_rdm_url = 'http://localhost:8001/'
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
weko_url = 'https://weko3.rdm.example.com/'
weko_user_email = None
weko_user_password = None
weko_institution_name = 'Massachusetts Institute of Technology [Test]'
weko_index_name = 'GRDM-Collaboration-Test-VR-1'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False
project_name = None
oauth_application_name = None
project_url = None
default_storage_label = 'NII Storage'

In [ ]:
import asyncio
import importlib
import os
import shutil
import tempfile
from urllib.parse import urljoin, urlparse

if idp_username_2 is None:
    prompt = f"Username for {idp_name_2 or 'user account'}: "
    idp_username_2 = input(prompt)
if idp_password_2 is None:
    prompt = f"Password for {idp_username_2}: "
    idp_password_2 = getpass(prompt)

if weko_user_email is None:
    weko_user_email = input('WEKO user email: ')
if weko_user_password is None:
    weko_user_password = getpass('WEKO user password: ')
if weko_institution_name is None:
    weko_institution_name = input('機関名 (例: GakuNin RDM IdP): ')
if weko_index_name is None:
    weko_index_name = 'TestJ'

if project_name is None:
    project_name = datetime.now().strftime('TEST-WEKO-%Y%m%d%H%M')
if oauth_application_name is None:
    oauth_application_name = datetime.now().strftime('TEST-WEKO-APP-%Y%m%d%H%M')



# プロジェクトに対するWEKOアドオンの登録

- サブシステム名: アドオン
- ページ/アドオン: WEKO
- 機能分類: プロジェクト設定
- シナリオ名: プロジェクトへの有効化
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM一般ユーザー, WEKO管理者、ユーザーadmin: GRDM統合管理者)
- 事前条件: 「機関でのJAIRO Cloud設定有効化」を実施済みであること

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()
    await grdm.expect_anonymous_toppage(page, idp_name_2, transition_timeout=transition_timeout)

await run_pw(_step, new_page=True)

## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## (利用規約とプライバシーポリシーの同意要求が表示された場合) 「これらの規約を読み、同意します」をチェックしたのち、「続ける」ボタンを押下する

利用規約とプライバシーポリシーに関する表示が消えること

In [ ]:
async def _step(page):
    checkbox = await page.query_selector('//input[@name="acceptedTermsOfService"]')
    if not checkbox:
        print(f"利用規約とプライバシーポリシーの同意要求は表示されませんでした。")
        return
    await checkbox.check()
    await page.click('//button[@data-analytics-name="Continue"]')
    await expect(page.locator('//input[@name="acceptedTermsOfService"]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
project_created = False

async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@full_name = "JAIRO Cloud"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「アドオンを選択」のパネル内「JAIRO Cloud」の行の「有効にする」をクリックする

「JAIRO Cloudアドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "JAIRO Cloud"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.click()
        await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
    else:
        print('JAIRO Cloud addon is already enabled for this project')

await run_pw(_step)


## 「確認」をクリックする

「アドオンを構成」のパネル内に「JAIRO Cloud」の行が追加されること

In [ ]:
async def _step(page):
    confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
    if await confirm_button.count():
        await confirm_button.click()
    await expect(page.locator('//img[@src = "/static/addons/weko/comicon.png"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


In [ ]:
connect_link_text = None


## 「アカウントに接続する」または「プロフィールからアカウントをインポート」リンクをクリックする

「アカウントに接続する」リンクの場合は「JAIRO Cloudアカウントに接続」ダイアログが、「プロフィールからアカウントをインポート」リンクの場合はWEKOアカウントに接続しますか？ダイアログが表示されること

In [ ]:
async def _step(page):
    trigger = page.locator('//img[@src = "/static/addons/weko/comicon.png"]/..//a[contains(text(), "アカウント")]')
    global connect_link_text
    connect_link_text = await trigger.inner_text()
    await trigger.click()
    if connect_link_text and 'アカウントに接続する' in connect_link_text:
        await expect(page.locator('//*[contains(@class, "modal-content")]//h3[contains(text(), "JAIRO Cloud")]')).to_be_visible(timeout=10000)
    else:
        await expect(page.locator('//*[contains(@class, "modal-content")]//h4[contains(text(), "JAIRO Cloud")]')).to_be_visible(timeout=10000)

await run_pw(_step)


##  表示されたリンクが「アカウントに接続する」の場合: WEKOリポジトリとして TEST-WEKO-APP-YYYYMMDDHHMM を選択する 

選択した値が表示されること



In [ ]:
async def _step(page):
    if connect_link_text and 'アカウントに接続する' in connect_link_text:
        selector = page.locator('//select[@id = "hostSelect"]')
        await expect(selector).to_be_visible(timeout=transition_timeout)
        await selector.select_option(label=oauth_application_name)
    else:
        print('Host selection is skipped because the profile import flow is used')

await run_pw(_step)


##  表示されたリンクが「アカウントに接続する」の場合: 「接続」をクリックし、WEKOログインページでWEKOログインを実施し、「Authorize application」をクリックする

GRDMのアドオン設定画面に戻り、「JAIRO Cloudアカウントに接続しますか？」ダイアログが表示されること


In [ ]:
async def _step(page):
    if connect_link_text and 'アカウントに接続する' in connect_link_text:
        popup_future = page.wait_for_event('popup')
        await page.locator('//button[@data-bind = "click: connectOAuth"]').click()
        popup = await popup_future
        await popup.locator('//input[@name = "email"]').fill(weko_user_email)
        await popup.locator('//input[@name = "password"]').fill(weko_user_password)
        await popup.locator('//button[@type = "submit"]').click()
        await popup.locator('//button[contains(text(), "Authorize application")]').click()
        await asyncio.sleep(10)  # wait for close
    else:
        print('OAuth popup is not required in this flow')

await run_pw(_step)


In [ ]:
async def _step(page):
    if connect_link_text and 'アカウントに接続する' in connect_link_text:
        await page.reload()
        await page.locator('//img[@src = "/static/addons/weko/comicon.png"]/..//a[contains(text(), "アカウント")]').click()
    await expect(page.locator('//*[contains(@class, "modal-content")]//button[text() = "接続" and @data-bb-handler="confirm"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


##  「接続」をクリックする

JAIRO Cloudアカウントに接続しますか？ダイアログが閉じること


In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "modal-content")]//button[text() = "接続" and @data-bb-handler="confirm"]').click()
    await expect(page.locator('//*[contains(text(), "authorized by")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


##  インデックス「TestJ」を選択する

ドロップダウンに TestJ と表示されること


In [ ]:
async def _step(page):
    select_box = page.locator('//div[contains(@class, "weko-settings")]//select')
    await select_box.select_option(label=weko_index_name)

await run_pw(_step)


##  「保存」をクリックする

現在のインデックス: Index for Testと表示されること


In [ ]:
async def _step(page):
    await page.locator('//div[contains(@class, "weko-settings")]//button[contains(text(), "保存")]').click()
    await expect(page.locator(f'//div[contains(@class, "weko-settings")]//a[contains(text(), "{weko_index_name}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「確認」をクリックする

「アドオンを構成」のパネル内に「Metadata」の行が追加され、かつ、上部メニューに「メタデータ」タブが追加されること

In [ ]:
async def _step(page):
    confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
    if await confirm_button.count():
        await confirm_button.click()
    await expect(page.locator('//a[contains(text(), "メタデータ")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「(プロジェクト名)」をクリックする

プロジェクトダッシュボードページが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//a[contains(text(), "{project_name}")]').click()
    await expect(page.locator(f'//*[text() = "{default_storage_label}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「JAIRO Cloud: (インデックス名)」をクリックする

「新規フォルダ作成」ボタンが表示されること (表示まで数秒かかる可能性がある)

In [ ]:
async def _step(page):
    target_label = f'JAIRO Cloud: {weko_index_name}'
    await page.locator(f'//*[text() = "{target_label}"]').click()
    await expect(page.locator('//*[text() = "新規フォルダ作成"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「新規フォルダ作成」をクリックする

フォルダ名入力テキストフィールドが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "新規フォルダ作成"]').click()
    await expect(page.locator('//input[@id = "createFolderInput"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「TESTFOLDER(ENTER)」を入力する

「TESTFOLDER [Draft]」と表示されること

In [ ]:
async def _step(page):
    folder_input = page.locator('//input[@id = "createFolderInput"]')
    await folder_input.fill('TESTFOLDER')
    await folder_input.press('Enter')
    await expect(page.locator('//*[text() = "TESTFOLDER"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「TESTFOLDER」のアイコンをクリックする

「フォルダを削除」ボタンが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "TESTFOLDER"]/../../..//i[contains(@class, "fa-folder")]').click()
    await expect(page.locator('//*[text() = "フォルダを削除"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「フォルダを削除」をクリックする

「"TESTFOLDER"を削除しますか？」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[text() = "フォルダを削除"]').click()
    await expect(page.locator('//h3[contains(text(), "削除しますか")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 「削除」をクリックする

TESTFOLDERフォルダが削除されること

In [ ]:
async def _step(page):
    await page.locator('//h3[contains(text(), "削除しますか")]/../..//*[contains(@class, "btn-danger") and text() = "削除"]').click()
    await expect(page.locator('//*[text() = "TESTFOLDER"]')).to_have_count(0)

await run_pw(_step)


後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)


In [ ]:
!rm -fr {work_dir}
